In [ ]:
## Imports 

import pandas as pd
import numpy as np
import os
import math
from src.cleaning.clean_stations import clean_station_data
from src.cleaning.merge_station_events import merge_station_events
from src.cleaning.merged_tornado_indicator import create_tornado_indicator



In [2]:
## Data Access and Setup
columns_to_drop = ['NAME',
                    'SOURCE',
                    'REPORT_TYPE',
                    'CALL_SIGN',
                    'QUALITY_CONTROL',
                    'CALL_SIGN.1',
                    'QUALITY_CONTROL.1',
                    'REPORT_TYPE.1',
                    'SOURCE.1',
                    'AB1',
                    'AD1',
                    'AE1',
                    'AG1',
                    'AH1',
                    'AH2',
                    'AH3',
                    'AH4',
                    'AH5',
                    'AH6',
                    'AI1',
                    'AI2',
                    'AI3',
                    'AI4',
                    'AI5',
                    'AI6',
                    'AK1',
                    'AM1',
                    'AN1',
                    'AT1',
                    'AT2',
                    'AT3',
                    'AT4',
                    'AT5',
                    'AT6',
                    'AT7',
                    'AT8',
                    'AU1',
                    'AU2',
                    'AU3',
                    'AU4',
                    'AU5',
                    'AW1',
                    'AW2',
                    'AW3',
                    'AW4',
                    'AW5',
                    'AW6',
                    'AW7',
                    'AX1',
                    'AX2',
                    'AX3',
                    'AX4',
                    'AX5',
                    'AX6',
                    'ED1',
                    'EQD',
                    'GD1',
                    'GD2',
                    'GD3',
                    'GD4',
                    'GE1',
                    'GF1',
                    'IA1',
                    'KC1',
                    'KC2',
                    'KD1',
                    'KD2',
                    'KE1',
                    'MH1',
                    'MK1',
                    'MV1',
                    'MW1',
                    'MW2',
                    'MW3',
                    'MW4',
                    'MW5',
                    'OD1',
                    'OE1',
                    'OE2',
                    'OE3',
                    'REM',
                    'SA1',
                    'UA1',
                    'UG1',
                    'WA1',
                    'AA1',
                    'AA2',
                    'AA3',
                    'AA4',
                    'AJ1',
                    'AL1',
                    'GA2',
                    'GA3',
                    'GA4',
                    'GA5',
                    'GA6',
                    'GJ1',
                    'GK1',
                    'GP1',
                    'GQ1',
                    'GR1',
                    'HL1',
                    'KA1',
                    'KA2',
                    'KA3',
                    'KA4',
                    'KB1',
                    'KB2',
                    'KB3',
                    'KG1',
                    'KG2',
                    'MD1',
                    'MF1',
                    'MG1',
                    'OC1',
                    'RH1',
                    'RH2',
                    'RH3'
                    ]
### Reasons to get rid of 
# 'NAME': already have an identifier column 'STATION'.
# 'SOURCE': this is just the source or sources used to create sample.
# 'REPORT_TYPE': denotes the type of geophysical surface observation.
# 'CALL_SIGN': call letters assigned to a weather station. We already have an identifier.
# 'QUALITY_CONTROl': For predicting tornadoes, this might not be super useful.
# though it may be good to keep in mind if so desired. One can erase all V01 
# entries (no quality control).
# 'CALL_SIGN.1': see CALL_SIGN.
# 'QUALITY_CONTROL.1' : see QUALITY_CONTROL.
# 'REPORT_TYPE.1': see REPORT_TYPE.
# 'SOURCE.1': see SOURCE.
# 'AB1': Liquid Precipitation Monthly total-- too long of a time scale.
# 'AD1: Liquid Precipitation Greatest Amount in 24 Hours, For the month -- too long of a time scale.
# 'AE1': Number of Days with Specific Amounts for Each Month -- Can be obtained through AA1-AA4
# 'AG1': 'Precipitation Estimated Observation -- not sure how this is different from AA1-AA4
# 'AH1'-- AH6' : Liquid Precipitation Maximum Short Duration, For The Month -- too long of a time scale.
# 'AI1 -- AI6' : Identical to 'AH1'--'AH6'
# 'AK1 Greatest Snow Depth on Ground for the Month
# 'AM1':
# 'AN1':
# 'AT1--AT8': Data leakage
# 'AU1--AU5': Data leakage 
# 'AW1--AW7': Data leakage
# 'AX1--AX6': Data leakage
# 'ED1': Runway Visibility
# 'EQD':
# 'GD1--GD4': Similar to GA1-GA6
# 'GE1': Similar to GA1-GA6 (may include later)
# 'GF1': Similar to GA1-GA6 (may include later)
# 'IA1': 
# 'KC1--KD2': too long of a time scale
# 'KE1': Extreme Temperatures, Number of Days Exceeding Criteria, For the Month -- too long of a time scale
# 'MH1': Atmospheric Pressure Observation for the month -- too long of a time scale.
# 'MK1' : See 'MH1'
# 'MV1 : Present Weather in Vicinity Observation -- Potential Data Leakage
# 'MW1--MW5' : Present Weather Observation -- Potential Data Leakage 
# 'OE1--OE3': Already have WND
# 'REM' : These are remarks
# 'SA1': Sea Surface Temperature 
# 'UA1'--'UG1' Marine Data?
# 'WA1'-- Platform Ice accretion ###


# SPLIT TUPLES IN PARTICULAR COLUMNS

# These are ordered in the way their tuples are ordered

mapping = {

'CIG': ['CIG- Sky Condition Observation- Ceiling Height Dimension',
       'CIG- Sky Condition Observation- Ceiling Quality Code',
       'CIG- Sky Condition Observation- Ceiling Determination Code',
       'CIG- Sky Condition Observation- Cavok Code'],

'DEW':['DEW- Air Temperature Observation- Dew Point Temperature',
       'DEW- Air Temperature Observation- Dew Point Quality Code'],

'GA1':['GA1- Sky Cover Layer- Coverage Code',
       'GA1- Sky Cover Layer- Coverage Quality Code',
       'GA1- Sky Cover Layer- Base Height Dimensions',
       'GA1- Sky Cover Layer- Base Height Quality Code',
       'GA1- Sky Cover Layer- Cloud Type Code',
       'GA1- Sky Cover Layer- Cloud Type Quality Code'],

'MA1':['MA1-Atmospheric Pressure Observation- Altimeter Setting Rate',
       'MA1-Atmospheric Pressure Observation- Altimeter Quality Code',
       'MA1-Atmospheric Pressure Observation- Station Pressure Rate',
       'MA1-Atmospheric Pressure Observation- Station Pressure Quality Code'],


'SLP':['SLP- Atmospheric Pressure Observation- Sea Level Pressure',
       'SLP- Atmospheric Pressure Observation- Sea Level Pressure Quality Code'],

'TMP':['TMP- Air Temperature Observation- Air Temperature',
       'TMP- Air Temperature Observation- Air Temperature Quality Code'],

'VIS': ['VIS- Visibility Observation- Distance Dimension',
       'VIS- Visibility Observation- Distance Quality Code',
       'VIS- Visibility Observation- Variability Code',
       'VIS- Visibility Observation- Quality Variability Code'],

'WND':['WND- Wind Observation- Direction Angle',
       'WND- Wind Observation- Direction Quality Code',
       'WND- Wind Observation- Type Code',
       'WND- Wind Observation- Speed Rate',
       'WND- Wind Observation- Speed Quality Code'],
}



data = create_tornado_indicator(drop_cols=columns_to_drop,split_tuples=True, mapping=mapping, tuple_sep=',', time_window=1,val_radius=50)

/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:89: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(f"{raw_dir}/{file}").copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:89: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(f"{raw_dir}/{file}").copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:89: DtypeWarning: Columns (1

In [4]:
## Data Access and Setup
# Sample
sample_of_data = data.sample(10)
sample_of_data

,STATION,STATION_LAT,STATION_LON,ELEVATION,MK1,OD1,YEAR-MONTH-DAY,STATION_TIME,AA1- Liquid Precipitation- Period Quantity in Hours,AA1- Liquid Precipitation- Depth Dimension,...,TORNADO_BEGIN_TIME,TORNADO_END_DATE,TORNADO_END_TIME,TORNADO_BEGIN_LAT,TORNADO_BEGIN_LON,TORNADO_END_LAT,TORNADO_END_LON,TORNADO_INITIAL_DISTANCE_FROM_STATION,TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km,TORNADO_OCCURRENCE
710690,72357003948,35.25000,-97.46667,360.3,NaN,NaN,2009-11-25,17:49:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1356634,72354013919,35.41667,-97.38333,393.5,NaN,NaN,2017-11-12,05:56:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1569174,72353013967,35.38843,-97.60035,389.9,NaN,NaN,2021-01-23,23:52:00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
256965,72354099999,35.41700,-97.38300,397.0,NaN,NaN,2003-09-12,18:00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49450,72353013967,35.38890,-97.60060,391.7,NaN,NaN,2000-11-26,12:53:00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1321343,72354403954,35.53417,-97.64694,395.3,NaN,NaN,2017-05-18,21:53:00,1.0,0.0,...,21:35:00,2017-05-18,21:40:00,35.5643,-95.3315,35.6049,-95.2516,209.499811,False,False
475216,72357003948,35.25000,-97.46667,360.3,NaN,NaN,2006-07-11,15:53:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29365,72354499999,35.53300,-97.65000,396.0,NaN,NaN,2000-07-09,22:53:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36593,72354099999,35.41700,-97.38300,397.0,NaN,NaN,2000-09-06,05:00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1426222,72353013967,35.38890,-97.60060,391.7,NaN,NaN,2019-01-02,17:52:00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
## Data Overview
# Record data.shape
data.shape

(1626352, 198)

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1626352 entries, 0 to 1626351
Columns: 198 entries, STATION to TORNADO_OCCURRENCE
dtypes: float64(131), int64(10), object(57)
memory usage: 2.4+ GB


In [13]:
data.select_dtypes(include='object').columns

Index(['MK1', 'OD1', 'YEAR-MONTH-DAY', 'STATION_TIME', 'AA1- Quality Code',
       'AJ1- Snow Depth- Quality Code',
       'CIG- Sky Condition Observation- Ceiling Determination Code',
       'CIG- Sky Condition Observation- Cavok Code',
       'DEW- Air Temperature Observation- Dew Point Quality Code',
       'GA1- Sky Cover Layer- Coverage Quality Code',
       'GA1- Sky Cover Layer- Base Height Quality Code',
       'GA1- Sky Cover Layer- Cloud Type Quality Code',
       'GA2- Sky Cover Layer- Coverage Quality Code',
       'GA2- Sky Cover Layer- Base Height Quality Code',
       'GA2- Sky Cover Layer- Cloud Type Quality Code',
       'GA3- Sky Cover Layer- Coverage Quality Code',
       'GA3- Sky Cover Layer- Base Height Quality Code',
       'GA3- Sky Cover Layer- Cloud Type Quality Code',
       'GA4- Sky Cover Layer- Coverage Quality Code',
       'GA4- Sky Cover Layer- Base Height Quality Code',
       'KA1- Extreme Air Temperature- Code',
       'KA1- Extreme Air Temperature- 

## **2. Data Overview**
* Note tornadoes are counted as occurring in a sample if the sample's station is within `val_radius`= 50 km away from the tornado's starting location and the station observes them within `time_window` hours of the tornado's beginning time. The variables `val_radius` and `time_window` are parameters that can be modified if desired. 

* **Shape of the Data**: `data` has 1626352 rows and 196
* **Number of Data Types** :
    * 131 floats
    * 10 ints
    * 55 objects

In [6]:
## Missing Values

# Some columns have particular missing value numbers as in station documentation

# Columns to check value counts if there are more than 80% NaN values
check_col_values =[]
# frequency of missing values 
na_freq = (data.isna().sum())/len(data)
for idx in na_freq.index:
    print(f'Feature `{idx}` has `{na_freq[idx]}` NaN frequency\n')
    if na_freq[idx]>=.5:
        check_col_values.append(idx)

Feature `STATION` has `0.0` NaN frequency

Feature `STATION_LAT` has `0.0` NaN frequency

Feature `STATION_LON` has `0.0` NaN frequency

Feature `ELEVATION` has `0.0` NaN frequency

Feature `AA1` has `0.68075422786703` NaN frequency

Feature `AA2` has `0.9770246539494525` NaN frequency

Feature `AA3` has `0.9980539268251891` NaN frequency

Feature `AA4` has `0.999998155380877` NaN frequency

Feature `AJ1` has `0.9918252629197123` NaN frequency

Feature `AL1` has `0.9966944425315061` NaN frequency

Feature `CIG` has `0.0` NaN frequency

Feature `DEW` has `0.0` NaN frequency

Feature `GA1` has `0.44041818745265476` NaN frequency

Feature `GA2` has `0.8382422747351127` NaN frequency

Feature `GA3` has `0.9332204836345391` NaN frequency

Feature `GA4` has `0.9982248615305912` NaN frequency

Feature `GA5` has `0.99991514752034` NaN frequency

Feature `GA6` has `0.9999944661426309` NaN frequency

Feature `GJ1` has `0.9990709268350271` NaN frequency

Feature `GK1` has `0.9990709268350271` NaN

Feature `STATION` has `0.0` NaN frequency

Feature `STATION_LAT` has `0.0` NaN frequency

Feature `STATION_LON` has `0.0` NaN frequency

Feature `ELEVATION` has `0.0` NaN frequency

Feature `MK1` has `0.9996636644465651` NaN frequency

Feature `OD1` has `0.9494752673467982` NaN frequency

Feature `YEAR-MONTH-DAY` has `0.0` NaN frequency

Feature `STATION_TIME` has `0.0` NaN frequency

Feature `AA1- Liquid Precipitation- Period Quantity in Hours` has `0.68075422786703` NaN frequency

Feature `AA1- Liquid Precipitation- Depth Dimension` has `0.68075422786703` NaN frequency

Feature `AA1- Liquid Precipitation- Condition Code` has `0.68075422786703` NaN frequency

Feature `AA1- Quality Code` has `0.68075422786703` NaN frequency

Feature `AA2- Liquid Precipitation- Period Quantity in Hours` has `0.9770246539494525` NaN frequency

Feature `AA2- Liquid Precipitation- Depth Dimension` has `0.9770246539494525` NaN frequency

Feature `AA2- Liquid Precipitation- Condition Code` has `0.9770246539494525` NaN frequency

Feature `AA2- Quality Code` has `0.9770246539494525` NaN frequency

Feature `AA3- Liquid Precipitation- Period Quantity in Hours` has `0.9980539268251891` NaN frequency

Feature `AA3- Liquid Precipitation- Depth Dimension` has `0.9980539268251891` NaN frequency

Feature `AA3- Liquid Precipitation- Condition Code` has `0.9980539268251891` NaN frequency

Feature `AA3- Quality Code` has `0.9980539268251891` NaN frequency

Feature `AA4- Liquid Precipitation- Period Quantity in Hours` has `0.999998155380877` NaN frequency

Feature `AA4- Liquid Precipitation- Depth Dimension` has `0.999998155380877` NaN frequency

Feature `AA4- Liquid Precipitation- Condition Code` has `0.999998155380877` NaN frequency

Feature `AA4- Quality Code` has `0.999998155380877` NaN frequency

Feature `AJ1- Snow Depth- Dimension` has `0.9918252629197123` NaN frequency

Feature `AJ1- Snow Depth- Condition Code` has `0.9918252629197123` NaN frequency

Feature `AJ1- Snow Depth- Quality Code` has `0.9918252629197123` NaN frequency

Feature `AJ1- Snow Depth- Equivalent Water Depth Dimension` has `0.9918252629197123` NaN frequency

Feature `AJ1- Snow Depth- Equivalent Water Condition Code` has `0.9918252629197123` NaN frequency

Feature `AJ1- Snow Depth- Equivalent Water Condition Quality Code` has `0.9918252629197123` NaN frequency

Feature `AL1- Snow Accumulation- Period Quantity` has `0.9966944425315061` NaN frequency

Feature `AL1- Snow Accumulation- Depth Dimension` has `0.9966944425315061` NaN frequency

Feature `AL1- Snow Accumulation- Condition Code` has `0.9966944425315061` NaN frequency

Feature `AL1- Snow Accumulation- Quality Code` has `0.9966944425315061` NaN frequency

Feature `CIG- Sky Condition Observation- Ceiling Height Dimension` has `0.0` NaN frequency

Feature `CIG- Sky Condition Observation- Ceiling Quality Code` has `0.0` NaN frequency

Feature `CIG- Sky Condition Observation- Ceiling Determination Code` has `0.0` NaN frequency

Feature `CIG- Sky Condition Observation- Cavok Code` has `0.0` NaN frequency

Feature `DEW- Air Temperature Observation- Dew Point Temperature` has `0.0` NaN frequency

Feature `DEW- Air Temperature Observation- Dew Point Quality Code` has `0.0` NaN frequency

Feature `GA1- Sky Cover Layer- Coverage Code` has `0.44041818745265476` NaN frequency

Feature `GA1- Sky Cover Layer- Coverage Quality Code` has `0.44041818745265476` NaN frequency

Feature `GA1- Sky Cover Layer- Base Height Dimensions` has `0.44041818745265476` NaN frequency

Feature `GA1- Sky Cover Layer- Base Height Quality Code` has `0.44041818745265476` NaN frequency

Feature `GA1- Sky Cover Layer- Cloud Type Code` has `0.44041818745265476` NaN frequency

Feature `GA1- Sky Cover Layer- Cloud Type Quality Code` has `0.44041818745265476` NaN frequency

Feature `GA2- Sky Cover Layer- Coverage Code` has `0.8382422747351127` NaN frequency

Feature `GA2- Sky Cover Layer- Coverage Quality Code` has `0.8382422747351127` NaN frequency

Feature `GA2- Sky Cover Layer- Base Height Dimensions` has `0.8382422747351127` NaN frequency

Feature `GA2- Sky Cover Layer- Base Height Quality Code` has `0.8382422747351127` NaN frequency

Feature `GA2- Sky Cover Layer- Cloud Type Code` has `0.8382422747351127` NaN frequency

Feature `GA2- Sky Cover Layer- Cloud Type Quality Code` has `0.8382422747351127` NaN frequency

Feature `GA3- Sky Cover Layer- Coverage Code` has `0.9332204836345391` NaN frequency

Feature `GA3- Sky Cover Layer- Coverage Quality Code` has `0.9332204836345391` NaN frequency

Feature `GA3- Sky Cover Layer- Base Height Dimensions` has `0.9332204836345391` NaN frequency

Feature `GA3- Sky Cover Layer- Base Height Quality Code` has `0.9332204836345391` NaN frequency

Feature `GA3- Sky Cover Layer- Cloud Type Code` has `0.9332204836345391` NaN frequency

Feature `GA3- Sky Cover Layer- Cloud Type Quality Code` has `0.9332204836345391` NaN frequency

Feature `GA4- Sky Cover Layer- Coverage Code` has `0.9982248615305912` NaN frequency

Feature `GA4- Sky Cover Layer- Coverage Quality Code` has `0.9982248615305912` NaN frequency

Feature `GA4- Sky Cover Layer- Base Height Dimensions` has `0.9982248615305912` NaN frequency

Feature `GA4- Sky Cover Layer- Base Height Quality Code` has `0.9982248615305912` NaN frequency

Feature `GA4- Sky Cover Layer- Cloud Type Code` has `0.9982248615305912` NaN frequency

Feature `GA4- Sky Cover Layer- Cloud Type Quality Code` has `0.9982248615305912` NaN frequency

Feature `GA5- Sky Cover Layer- Coverage Code` has `0.99991514752034` NaN frequency

Feature `GA5- Sky Cover Layer- Coverage Quality Code` has `0.99991514752034` NaN frequency

Feature `GA5- Sky Cover Layer- Base Height Dimensions` has `0.99991514752034` NaN frequency

Feature `GA5- Sky Cover Layer- Base Height Quality Code` has `0.99991514752034` NaN frequency

Feature `GA5- Sky Cover Layer- Cloud Type Code` has `0.99991514752034` NaN frequency

Feature `GA5- Sky Cover Layer- Cloud Type Quality Code` has `0.99991514752034` NaN frequency

Feature `GA6- Sky Cover Layer- Coverage Code` has `0.9999944661426309` NaN frequency

Feature `GA6- Sky Cover Layer- Coverage Quality Code` has `0.9999944661426309` NaN frequency

Feature `GA6- Sky Cover Layer- Base Height Dimensions` has `0.9999944661426309` NaN frequency

Feature `GA6- Sky Cover Layer- Base Height Quality Code` has `0.9999944661426309` NaN frequency

Feature `GA6- Sky Cover Layer- Cloud Type Code` has `0.9999944661426309` NaN frequency

Feature `GA6- Sky Cover Layer- Cloud Type Quality Code` has `0.9999944661426309` NaN frequency

Feature `GJ1- Sunshine Observation- Sunshine Duration Quantity` has `0.9990709268350271` NaN frequency

Feature `GJ1- Sunshine Observation- Sunshine Duration Quality Code` has `0.9990709268350271` NaN frequency

Feature `GK1- Sunshine Observation- Percent of Possible Sunshine Quantity` has `0.9990709268350271` NaN frequency

Feature `GK1- Sunshine Observation- Percent of Possible Sunshine Quality Code` has `0.9990709268350271` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Time Period in Minutes` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Source Flag` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Uncertainty` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Direct Normal` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Source Flag` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Uncertainty` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal` has `0.9092490432575482` NaN frequency

Feature `GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal Source Flag` has `0.9092490432575482` NaN frequency

Feature `GQ1- Hourly Solar Angle Section- Hourly Solar Angle Time Period` has `0.9499394964927642` NaN frequency

Feature `GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle` has `0.9499394964927642` NaN frequency

Feature `GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle Quality Code` has `0.9499394964927642` NaN frequency

Feature `GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle` has `0.9499394964927642` NaN frequency

Feature `GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle Quality Code` has `0.9499394964927642` NaN frequency

Feature `GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Time Period` has `0.9092490432575482` NaN frequency

Feature `GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface` has `0.9092490432575482` NaN frequency

Feature `GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface Quality Code` has `0.9092490432575482` NaN frequency

Feature `GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun` has `0.9092490432575482` NaN frequency

Feature `GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun Quality Code` has `0.9092490432575482` NaN frequency

Feature `HL1- Hail- Size` has `0.9999889322852618` NaN frequency

Feature `HL1- Hail- Size Quality Code` has `0.9999889322852618` NaN frequency

Feature `KA1- Extreme Air Temperature- Period Quantity` has `0.9036106574714453` NaN frequency

Feature `KA1- Extreme Air Temperature- Code` has `0.9036106574714453` NaN frequency

Feature `KA1- Extreme Air Temperature- Air Temperature` has `0.9036106574714453` NaN frequency

Feature `KA1- Extreme Air Temperature- Temperature Quality Code` has `0.9036106574714453` NaN frequency

Feature `KA2- Extreme Air Temperature- Period Quantity` has `0.9042249156394188` NaN frequency

Feature `KA2- Extreme Air Temperature- Code` has `0.9042249156394188` NaN frequency

Feature `KA2- Extreme Air Temperature- Air Temperature` has `0.9042249156394188` NaN frequency

Feature `KA2- Extreme Air Temperature- Temperature Quality Code` has `0.9042249156394188` NaN frequency

Feature `KA3- Extreme Air Temperature- Period Quantity` has `0.9931478548309345` NaN frequency

Feature `KA3- Extreme Air Temperature- Code` has `0.9931478548309345` NaN frequency

Feature `KA3- Extreme Air Temperature- Air Temperature` has `0.9931478548309345` NaN frequency

Feature `KA3- Extreme Air Temperature- Temperature Quality Code` has `0.9931478548309345` NaN frequency

Feature `KA4- Extreme Air Temperature- Period Quantity` has `0.9932271734532254` NaN frequency

Feature `KA4- Extreme Air Temperature- Code` has `0.9932271734532254` NaN frequency

Feature `KA4- Extreme Air Temperature- Air Temperature` has `0.9932271734532254` NaN frequency

Feature `KA4- Extreme Air Temperature- Temperature Quality Code` has `0.9932271734532254` NaN frequency

Feature `KB1- Average Air Temperature- Period Quantity` has `0.9996648941926471` NaN frequency

Feature `KB1- Average Air Temperature- Type Code` has `0.9996648941926471` NaN frequency

Feature `KB1- Average Air Temperature- Air Temperature` has `0.9996648941926471` NaN frequency

Feature `KB1- Average Air Temperature- Temperature Quality Code` has `0.9996648941926471` NaN frequency

Feature `KB2- Average Air Temperature- Period Quantity` has `0.9996648941926471` NaN frequency

Feature `KB2- Average Air Temperature- Type Code` has `0.9996648941926471` NaN frequency

Feature `KB2- Average Air Temperature- Air Temperature` has `0.9996648941926471` NaN frequency

Feature `KB2- Average Air Temperature- Temperature Quality Code` has `0.9996648941926471` NaN frequency

Feature `KB3- Average Air Temperature- Period Quantity` has `0.9996648941926471` NaN frequency

Feature `KB3- Average Air Temperature- Type Code` has `0.9996648941926471` NaN frequency

Feature `KB3- Average Air Temperature- Air Temperature` has `0.9996648941926471` NaN frequency

Feature `KB3- Average Air Temperature- Temperature Quality Code` has `0.9996648941926471` NaN frequency

Feature `KG1- Average Dew Point and Wet Bulb Temperature- Period Quantity` has `0.9958323905280038` NaN frequency

Feature `KG1- Average Dew Point and Wet Bulb Temperature- Code` has `0.9958323905280038` NaN frequency

Feature `KG1- Average Dew Point and Wet Bulb Temperature- Temperature` has `0.9958323905280038` NaN frequency

Feature `KG1- Average Dew Point and Wet Bulb Temperature- Derived Code` has `0.9958323905280038` NaN frequency

Feature `KG1- Average Dew Point and Wet Bulb Temperature- Quality Code` has `0.9958323905280038` NaN frequency

Feature `KG2- Average Dew Point and Wet Bulb Temperature- Period Quantity` has `0.9958323905280038` NaN frequency

Feature `KG2- Average Dew Point and Wet Bulb Temperature- Code` has `0.9958323905280038` NaN frequency

Feature `KG2- Average Dew Point and Wet Bulb Temperature- Temperature` has `0.9958323905280038` NaN frequency

Feature `KG2- Average Dew Point and Wet Bulb Temperature- Derived Code` has `0.9958323905280038` NaN frequency

Feature `KG2- Average Dew Point and Wet Bulb Temperature- Quality Code` has `0.9958323905280038` NaN frequency

Feature `MA1-Atmospheric Pressure Observation- Altimeter Setting Rate` has `0.07189710468582447` NaN frequency

Feature `MA1-Atmospheric Pressure Observation- Altimeter Quality Code` has `0.07189710468582447` NaN frequency

Feature `MA1-Atmospheric Pressure Observation- Station Pressure Rate` has `0.07189710468582447` NaN frequency

Feature `MA1-Atmospheric Pressure Observation- Station Pressure Quality Code` has `0.07189710468582447` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Tendency Code` has `0.8208801046759865` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Quality Tendency Code` has `0.8208801046759865` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Three Hour Quantity` has `0.8208801046759865` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Quality Three Hour Code` has `0.8208801046759865` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Twenty Four Hour Quantity` has `0.8208801046759865` NaN frequency

Feature `MD1- Atmospheric Pressure Change- Quality Twenty Four Hour Code` has `0.8208801046759865` NaN frequency

Feature `MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)` has `0.9958606746878904` NaN frequency

Feature `MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code` has `0.9958606746878904` NaN frequency

Feature `MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure for the Day` has `0.9958606746878904` NaN frequency

Feature `MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure Quality Code` has `0.9958606746878904` NaN frequency

Feature `MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)` has `0.9883942713508514` NaN frequency

Feature `MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code` has `0.9883942713508514` NaN frequency

Feature `MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure for the Day` has `0.9883942713508514` NaN frequency

Feature `MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure Quality Code` has `0.9883942713508514` NaN frequency

Feature `OC1- Wind Gust Observation- Speed Rate` has `0.7830881629561128` NaN frequency

Feature `OC1- Wind Gust Observation- Quality Code` has `0.7830881629561128` NaN frequency

Feature `RH1- Relative Humidity- Period Quantity` has `0.9958096402254862` NaN frequency

Feature `RH1- Relative Humidity- Code` has `0.9958096402254862` NaN frequency

Feature `RH1- Relative Humidity- Percentage` has `0.9958096402254862` NaN frequency

Feature `RH1- Relative Humidity- Derived Code` has `0.9958096402254862` NaN frequency

Feature `RH1- Relative Humidity- Quality Code` has `0.9958096402254862` NaN frequency

Feature `RH2- Relative Humidity- Period Quantity` has `0.9958096402254862` NaN frequency

Feature `RH2- Relative Humidity- Code` has `0.9958096402254862` NaN frequency

Feature `RH2- Relative Humidity- Percentage` has `0.9958096402254862` NaN frequency

Feature `RH2- Relative Humidity- Derived Code` has `0.9958096402254862` NaN frequency

Feature `RH2- Relative Humidity- Quality Code` has `0.9958096402254862` NaN frequency

Feature `RH3- Relative Humidity- Period Quantity` has `0.9958096402254862` NaN frequency

Feature `RH3- Relative Humidity- Code` has `0.9958096402254862` NaN frequency

Feature `RH3- Relative Humidity- Percentage` has `0.9958096402254862` NaN frequency

Feature `RH3- Relative Humidity- Derived Code` has `0.9958096402254862` NaN frequency

Feature `RH3- Relative Humidity- Quality Code` has `0.9958096402254862` NaN frequency

Feature `SLP- Atmospheric Pressure Observation- Sea Level Pressure` has `0.0` NaN frequency

Feature `SLP- Atmospheric Pressure Observation- Sea Level Pressure Quality Code` has `0.0` NaN frequency

Feature `TMP- Air Temperature Observation- Air Temperature` has `0.0` NaN frequency

Feature `TMP- Air Temperature Observation- Air Temperature Quality Code` has `0.0` NaN frequency

Feature `VIS- Visibility Observation- Distance Dimension` has `0.0` NaN frequency

Feature `VIS- Visibility Observation- Distance Quality Code` has `0.0` NaN frequency

Feature `VIS- Visibility Observation- Variability Code` has `0.0` NaN frequency

Feature `VIS- Visibility Observation- Quality Variability Code` has `0.0` NaN frequency

Feature `WND- Wind Observation- Direction Angle` has `0.0` NaN frequency

Feature `WND- Wind Observation- Direction Quality Code` has `0.0` NaN frequency

Feature `WND- Wind Observation- Type Code` has `0.0` NaN frequency

Feature `WND- Wind Observation- Speed Rate` has `0.0` NaN frequency

Feature `WND- Wind Observation- Speed Quality Code` has `0.0` NaN frequency

Feature `TORNADO_BEGIN_DATE` has `0.8023287701555383` NaN frequency

Feature `TORNADO_BEGIN_TIME` has `0.8023287701555383` NaN frequency

Feature `TORNADO_END_DATE` has `0.8023287701555383` NaN frequency

Feature `TORNADO_END_TIME` has `0.8023287701555383` NaN frequency

Feature `TORNADO_BEGIN_LAT` has `0.8030494013596072` NaN frequency

Feature `TORNADO_BEGIN_LON` has `0.8030494013596072` NaN frequency

Feature `TORNADO_END_LAT` has `0.8030494013596072` NaN frequency

Feature `TORNADO_END_LON` has `0.8030494013596072` NaN frequency

Feature `TORNADO_INITIAL_DISTANCE_FROM_STATION` has `0.8030494013596072` NaN frequency

Feature `TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km` has `0.8030494013596072` NaN frequency

Feature `TORNADO_OCCURRENCE` has `0.8023287701555383` NaN frequency

In [41]:
check_col_values

['MK1',
 'OD1',
 'AA2- Liquid Precipitation- Period Quantity in Hours',
 'AA2- Liquid Precipitation- Depth Dimension',
 'AA2- Liquid Precipitation- Condition Code',
 'AA2- Quality Code',
 'AA3- Liquid Precipitation- Period Quantity in Hours',
 'AA3- Liquid Precipitation- Depth Dimension',
 'AA3- Liquid Precipitation- Condition Code',
 'AA3- Quality Code',
 'AA4- Liquid Precipitation- Period Quantity in Hours',
 'AA4- Liquid Precipitation- Depth Dimension',
 'AA4- Liquid Precipitation- Condition Code',
 'AA4- Quality Code',
 'AJ1- Snow Depth- Dimension',
 'AJ1- Snow Depth- Condition Code',
 'AJ1- Snow Depth- Quality Code',
 'AJ1- Snow Depth- Equivalent Water Depth Dimension',
 'AJ1- Snow Depth- Equivalent Water Condition Code',
 'AJ1- Snow Depth- Equivalent Water Condition Quality Code',
 'AL1- Snow Accumulation- Period Quantity',
 'AL1- Snow Accumulation- Depth Dimension',
 'AL1- Snow Accumulation- Condition Code',
 'AL1- Snow Accumulation- Quality Code',
 'GA2- Sky Cover Layer- Covera

In [4]:
for col in check_col_values:
    display(data[col].value_counts())

AA2
06,0000,2,1    6127
03,0000,2,1    5389
06,0000,9,1    4614
06,0003,9,1    1387
03,0003,9,1    1249
               ... 
24,0681,3,1       1
06,0676,9,1       1
24,0183,3,1       1
24,0272,3,1       1
24,0295,3,1       1
Name: count, Length: 1026, dtype: int64

AA3
24,0003,9,1    202
24,0005,9,1    138
24,0008,9,1    131
24,0010,9,1     91
24,0023,9,1     71
              ... 
24,0803,9,1      1
24,0048,9,2      1
24,0681,9,1      1
24,1646,9,1      1
24,1128,9,1      1
Name: count, Length: 304, dtype: int64

AA4
06,9999,2,9    3
Name: count, dtype: int64

AJ1
0000,9,5,999999,9,9    11715
0000,9,I,999999,9,9      385
0003,9,1,999999,9,9      133
0003,9,5,999999,9,9      113
0005,9,1,999999,9,9       88
                       ...  
0000,3,1,000000,9,9        1
0036,9,P,999999,9,9        1
0003,9,1,000250,9,9        1
0010,3,1,001020,9,9        1
0003,9,5,000300,9,9        1
Name: count, Length: 94, dtype: int64

AL1
24,000,9,5    4999
24,000,9,6     254
24,000,3,5      77
01,999,9,9      12
99,001,3,1       9
24,002,9,5       4
99,000,9,1       4
24,002,9,6       2
24,001,9,6       2
24,001,9,5       2
99,008,3,1       2
99,012,3,1       1
99,004,3,1       1
99,007,3,1       1
24,999,9,9       1
99,002,3,1       1
99,005,3,1       1
24,003,9,6       1
24,006,9,6       1
99,003,3,1       1
Name: count, dtype: int64

...

TORNADO_END_LAT
35.33333    1784
35.28333    1577
35.30000    1436
35.50000    1378
35.25000    1305
            ... 
36.68690     127
36.65270     127
34.20000     126
35.36060     124
34.11667      90
Name: count, Length: 1321, dtype: int64

TORNADO_END_LON
-95.38333    1231
-97.45000    1128
-99.10000     914
-96.50000     818
-96.40000     807
             ... 
-94.75000     122
-96.23333     122
-95.31667     122
-98.40000      90
-98.01667      90
Name: count, Length: 1402, dtype: int64

TORNADO_INITIAL_DISTANCE_FROM_STATION
143.603126    280
142.381269    275
65.495436     241
157.657285    216
151.975364    201
             ... 
166.871560      1
173.587409      1
205.358728      1
169.157800      1
93.840821       1
Name: count, Length: 6692, dtype: int64

TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km
False    285441
True      34870
Name: count, dtype: int64

TORNADO_OCCURRENCE
False    317148
True       4335
Name: count, dtype: int64

In [7]:
check_col_values

['AA1',
 'AA2',
 'AA3',
 'AA4',
 'AJ1',
 'AL1',
 'GA2',
 'GA3',
 'GA4',
 'GA5',
 'GA6',
 'GJ1',
 'GK1',
 'GP1',
 'GQ1',
 'GR1',
 'HL1',
 'KA1',
 'KA2',
 'KA3',
 'KA4',
 'KB1',
 'KB2',
 'KB3',
 'KG1',
 'KG2',
 'MD1',
 'MF1',
 'MG1',
 'OC1',
 'RH1',
 'RH2',
 'RH3',
 'TORNADO_BEGIN_DATE',
 'TORNADO_BEGIN_TIME',
 'TORNADO_END_DATE',
 'TORNADO_END_TIME',
 'TORNADO_BEGIN_LAT',
 'TORNADO_BEGIN_LON',
 'TORNADO_END_LAT',
 'TORNADO_END_LON',
 'TORNADO_INITIAL_DISTANCE_FROM_STATION',
 'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km',
 'TORNADO_OCCURRENCE']

In [28]:
data['GJ1- Sunshine Observation- Sunshine Duration Quantity'].value_counts()

GJ1- Sunshine Observation- Sunshine Duration Quantity
0.0    1511
Name: count, dtype: int64